In [1]:
import random
import pandas as pd
from transformers import AutoModel, AutoTokenizer
from transformers import PreTrainedTokenizer
import os
import sys
from tensor2tensor.data_generators import text_encoder
import importlib
from typing import List, Tuple, Dict, Iterable, Optional
import inspect
import re
import unicodedata as ud
import pandas as pd
import torch
from torch.utils.data import DataLoader
from typing import Iterable, Optional
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import json

/home/jupyter-vojta/notebooks/labyrinth/venv_torch_nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-01 23:36:24.016836: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-01 23:36:24.068402: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# --- runtime hygiene for notebooks ---
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")  # stop fork warnings
os.environ.setdefault("OMP_NUM_THREADS", "32")
os.environ.setdefault("MKL_NUM_THREADS", "32")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "32")

# Optional: safer multiprocessing on some setups
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

## Loading WSD annotated data

In [3]:
data = []
with open("/srv/data/latin/LatinWSD/data/semeval_wsd.data", encoding="utf-8") as f:
    for i, line in enumerate(f):
        fields = line.rstrip('\n').split('\t', 4)
        if len(fields) == 5:
            data.append(fields)
        else:
            print(f"Malformed line at {i+1}: {fields}")

import pandas as pd
semeval_wsd = pd.DataFrame(
    data,
    columns=['lemma', 'sense_id', 'left_context', 'target_word', 'right_context']
)
semeval_wsd.head()

,lemma,sense_id,left_context,target_word,right_context
0,acerbus,III,", </line><line> ubi pudendum est ibi eos deser...",acerbior,Herculi quam illa mihi obiectast. </line><lin...
1,acerbus,I,", a nonnullis quaeritur. Beatus uero Augustinu...",acerbam,et dentes filiorum obstupuerunt;. Item: Non e...
2,acerbus,III,", cum ex aedito loco ingentem illam hominum mu...",acerbius,"? Quid tristius expectetur, etiamsi Phaethonte..."
3,acerbus,III,", cum interea quis vestrum hoc non audivit, qu...",acerbissimum,extat indicium et ad insignem memoriam turpit...
4,acerbus,III,", cum validis lucta certabat. Tum ad Syllam it...",acerbe,"in eum invectus fuisset, ad supplicium duci j..."


In [4]:
len(semeval_wsd)

2396

In [5]:
silver_inter_wsd = pd.read_csv("/srv/data/latin/LatinWSD/data/silver_align_wsd.data",
    sep="\t",
    names=['lemma', 'sense_id', 'left_context', 'target_word', 'right_context'],
    header=None,
    encoding="utf-8"
)

In [6]:
silver_inter_wsd.sample(10)

,lemma,sense_id,left_context,target_word,right_context
3323,imperator,I,NaN,imperatores,"enim comitiis consularibus , non verborum inte..."
887,consilium,III,"huc concessero , dum mihi senatum",consili,in cor convoco .
6005,senatus,I,centuripinorum,senatus,"decrevit populusque iussit ut , quae statuae v..."
4785,senatus,I,"clarissimus et fortissimus vir consul , inimic...",senatus,", defensor vestrae voluntatis , patronus publi..."
2781,hostis,II,estne hic,hostis,", quem aspicio , meus ?"
5612,senatus,I,nec vero ulla in re memini aut,senatum,meliorem aut magistratus ;
3246,imperator,I,denique usi liberalitate antoni milites impera...,imperatori,suo quam antonius etiam conlegae parentasset .
5491,senatus,I,ut enim primum post antoni foedissimum discessum,senatus,"haberi libere potuit , ad illum animum meum re..."
5416,senatus,I,m . cicero s . d . p . lentvlo procos . idibus...,senatu,"nihil est confectum , propterea quod dies magn..."
4677,sapientia,III,quae tamen ego omnia in expetenda amicitia tua...,sapientia,secutus sum .


In [7]:
latbert_sense_data = pd.read_csv(
    "/home/jupyter-vojta/notebooks/latin-bert/case_studies/wsd/data/latin.sense.data",
    sep="\t",
    names=['lemma', 'sense_id', 'left_context', 'target_word', 'right_context'],
    header=None,
    encoding="utf-8"
)

In [8]:
latbert_sense_data.head(10)

,lemma,sense_id,left_context,target_word,right_context
0,ab,I,caesar maturat,ab,urbe proficisci
1,ab,I,illam ( mulierem ) usque,a,mari supero romam proficisci
2,ab,I,quemadmodum ( caesar ),a,gergovia discederet
3,ab,I,protinus,a,corfinio in siciliam miserat
4,ab,I,quasi ad adulescentem,a,patre ex seleucia veniat
5,ab,I,nigidium,a,domitio capuam venisse
6,ab,I,ego te afuisse tam diu,a,nobis dolui
7,ab,I,abesse,a,domo paulisper maluit
8,ab,I,tum brutus,ab,roma aberat
9,ab,I,quot milia fundus suus abesset,ab,urbe


In [9]:
latbert_sense_data["wsd_source"] = "labert_wsd"
semeval_wsd["wsd_source"] = "semeval_wsd"
silver_inter_wsd["wsd_source"] = "silver_inter_wsd"

wsd_data = pd.concat([latbert_sense_data, semeval_wsd, silver_inter_wsd])

In [10]:
TEXT_COLS = ["left_context", "target_word", "right_context"]

TAG_RE = re.compile(r"<[^>]+>")

# Latin + extended letters (covers Latin diacritics). We normalize to NFC, so combining marks are folded.
LETTER = r"A-Za-zÀ-ÖØ-öø-ÿĀ-žḀ-ỿ"

# Delete these always (they cause phantom splits): ZWSP/ZWNJ/ZWJ/NO-BREAK BOM/soft hyphen
ZERO_WIDTH_RE = re.compile(r"[\u200B\u200C\u200D\u2060\uFEFF\u00AD]")

# Delete non-letters that are *between* letters (hyphenation debris, OCR joiners)
NONLETTER_BETWEEN_LETTERS = re.compile(fr"(?<=[{LETTER}])[^\s{LETTER}]+(?=[{LETTER}])")

# Replace all other remaining non-letters with a space
NONLETTER_TO_SPACE = re.compile(fr"[^{LETTER}\s]+")

# Optional: split lower→UPPER boundaries only (kept OFF by default for safety)
LOWER_TO_UPPER_RE  = re.compile(r"(?<=[a-zà-öø-ÿā-žḀ-ỿ])(?=[A-Z])")

def sanitize_text_columns(
    df: pd.DataFrame,
    cols: Iterable[str] = TEXT_COLS,
    *,
    split_camel: bool = False,   # ← default False to avoid unexpected splits
    drop_tags: bool = True,
    collapse_ws: bool = True,
    lower: bool = False,
) -> pd.DataFrame:
    def _clean(s: Optional[str]) -> str:
        if not isinstance(s, str):
            s = "" if s is None else str(s)

        # 0) NFC normalize to fold combining marks
        s = ud.normalize("NFC", s)

        # 1) strip tags
        if drop_tags:
            s = TAG_RE.sub(" ", s)

        # 2) kill zero-width / soft hyphen outright
        s = ZERO_WIDTH_RE.sub("", s)

        # 3) delete non-letters only when sandwiched by letters (fixes BRITA­NNI​CUS → BRITANNICUS)
        s = NONLETTER_BETWEEN_LETTERS.sub("", s)

        # 4) all other non-letters become spaces (preserves word boundaries)
        s = NONLETTER_TO_SPACE.sub(" ", s)

        # 5) optional camel split (safe: only lower→UPPER)
        if split_camel:
            s = LOWER_TO_UPPER_RE.sub(" ", s)

        # 6) whitespace + optional lower
        if collapse_ws:
            s = re.sub(r"\s+", " ", s).strip()
        if lower:
            s = s.lower()
        return s

    out = df.copy()
    out[list(cols)] = out[list(cols)].fillna("")
    for c in cols:
        out[c] = out[c].map(_clean)
    return out


In [11]:
# Clean all three datasets identically
# can also add lower=True to lowercase all columns
wsd_data = sanitize_text_columns(wsd_data, lower=True)

In [12]:
def drop_missing_sense(df: pd.DataFrame, sense_col: str = "sense_id") -> pd.DataFrame:
    out = df.copy()
    # Normalize to string, strip, and filter
    s = out[sense_col].astype(str).str.strip()
    mask_ok = (s.ne("")) & (s.str.lower().ne("nan"))
    dropped = (~mask_ok).sum()
    if dropped:
        print(f"Dropping {dropped} rows with empty/NaN {sense_col}.")
    return out[mask_ok].reset_index(drop=True)

# Now drop rows missing sense labels
wsd_data  = drop_missing_sense(wsd_data)

In [13]:
wsd_data[~wsd_data["sense_id"].isin(["I", "II", "III", "IV", "V", "VI", "VII"])]

,lemma,sense_id,left_context,target_word,right_context,wsd_source


In [14]:
wsd_data["left_context"] = wsd_data["left_context"].apply(lambda x: " ".join(x.split()[-10:]))
wsd_data["right_context"] = wsd_data["right_context"].apply(lambda x: " ".join(x.split()[:10]))

In [15]:
wsd_data.to_parquet("../data/wsd_data/wsd_data.parquet")

#latbert_sense_data.to_parquet("../data/wsd_data/labert_sense_data.parquet")
#semeval_wsd.to_parquet("../data/wsd_data/semeval_wsd.parquet")
#silver_inter_wsd.to_parquet("../data/wsd_data/silver_inter_wsd.parquet")

## Preparing for training

In [16]:
# load models

DEVICE = "cpu"
MODEL_ID = "xlm-roberta-base"
tokenizer_xlmr = AutoTokenizer.from_pretrained(MODEL_ID)
model_xlmr     = AutoModel.from_pretrained(
                MODEL_ID,
                output_hidden_states=True,
                output_attentions=True
            ).to(DEVICE).eval()

In [17]:
base_path = "/srv/models/latin-bert"
tok_path  = os.path.join(base_path, "latin_tokenizer.py")

# Force reload
spec = importlib.util.spec_from_file_location("latin_tokenizer", tok_path)
latin_tokenizer = importlib.util.module_from_spec(spec)
sys.modules["latin_tokenizer"] = latin_tokenizer  # replace any existing module
spec.loader.exec_module(latin_tokenizer)

# Re-instantiate the class with your encoder + vocab
LatinTokenizer = latin_tokenizer.LatinTokenizer
vocab_file_path = "/srv/models/latin-bert/vocab.txt"
subword_tokenizer_path = "/srv/models/latin-bert/latin.subword.encoder"

encoder = text_encoder.SubwordTextEncoder(subword_tokenizer_path)
tokenizer_labert = LatinTokenizer(vocab_file_path, encoder)

LABERT_BASE_DIR= "/srv/models/latin-bert"
model_labert = AutoModel.from_pretrained(LABERT_BASE_DIR).to(DEVICE).eval()

In [18]:
@torch.no_grad()
def has_nan_params(enc):
    for _, p in enc.named_parameters():
        if p.numel() and not torch.isfinite(p).all():
            return True
    return False

print("params clean?", not has_nan_params(model_labert))

params clean? True


In [19]:
# ========= helpers (string-only; no GreLa token dicts needed) =========
MAX_LEN = 256

def _ws(s: str) -> List[str]:
    return s.strip().split()

def encode_trunc(text: str, tokenizer, device="cpu", max_len=512):
    kwargs = {
        "text": text,
        "return_tensors": "pt",
        "truncation": True,
        "max_length": max_len,
    }
    # Check for optional argument support
    sig = inspect.signature(tokenizer.__call__)
    if "add_special_tokens" in sig.parameters:
        kwargs["add_special_tokens"] = True

    result = tokenizer(**kwargs)
    return result.to(device) if hasattr(result, "to") else result



def _tokens_and_target_idx(left: str, target: str, right: str) -> Tuple[List[str], int]:
    lt = _ws(left); rt = _ws(right)
    return lt + [target] + rt, len(lt)

# ---- XLM-R path: uses enc.word_ids() on the BatchEncoding (fast tokenizers) ----
def span_xlmr(tokenizer, tokens: List[str], target_idx: int, max_length: int = MAX_LEN) -> List[int]:
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                    padding="max_length", truncation=True, max_length=max_length)
    word_ids = enc.word_ids(0)  # <-- on enc, not tokenizer
    return [] if word_ids is None else [i for i, wid in enumerate(word_ids) if wid == target_idx]

# ---- LaBERT path: string-only; uses tokenizer.encode + build_inputs_with_special_tokens ----
def _first_occurrence(haystack: List[int], needle: List[int]) -> Optional[int]:
    """Find the first start index of 'needle' subsequence in 'haystack'; return None if not found."""
    if not needle:
        return None
    n, m = len(haystack), len(needle)
    for i in range(0, n - m + 1):
        if haystack[i:i+m] == needle:
            return i
    return None

def span_labert(tokenizer, tokens: List[str], target_idx: int, max_length: int = MAX_LEN) -> List[int]:
    # 1) subword length per whitespace token (no specials)
    wp_lens: List[int] = []
    for tok in tokens:
        try:
            ids = tokenizer.encode(tok, add_special_tokens=False)
        except TypeError:
            # if add_special_tokens is not supported, encode without it (most custom tokenizers do plain encode)
            ids = tokenizer.encode(tok)
        wp_lens.append(len(ids))

    # 2) build a full encoded input for the sentence (with specials/padding) to get seq_len for clipping
    sent_str = " ".join(tokens)
    enc = encode_trunc(sent_str, tokenizer=tokenizer, device="cpu", max_len=max_length)  # your working helper
    seq_len = int(enc["attention_mask"][0].sum().item())

    # 3) estimate left specials robustly
    try:
        core_ids = tokenizer.encode(sent_str, add_special_tokens=False)
    except TypeError:
        # if add_special_tokens kwarg not supported, try plain encode and then strip specials via build_inputs helper
        core_ids = tokenizer.encode(sent_str)  # may include specials; we’ll still handle below

    left_specials = 0
    try:
        # Prefer tokenizer's own builder if available
        built = tokenizer.build_inputs_with_special_tokens(core_ids)
        start = _first_occurrence(built, core_ids)
        left_specials = start if start is not None else 1
    except Exception:
        # Fallback: build a version with specials via encode(add_special_tokens=True) and align
        try:
            with_spec = tokenizer.encode(sent_str, add_special_tokens=True)
            start = _first_occurrence(with_spec, core_ids)
            left_specials = start if start is not None else 1
        except Exception:
            left_specials = 1  # conservative default (CLS/<s>)

    # 4) compute target subword span indices in the final sequence
    start_in_core = sum(wp_lens[:target_idx])
    k = wp_lens[target_idx] if target_idx < len(wp_lens) else 0
    start = left_specials + start_in_core
    end = start + max(0, k) - 1
    if k <= 0:
        return []
    return [i for i in range(start, end + 1) if 0 <= i < seq_len]

def add_nav_columns_after_merge(
    df,
    tok_xlmr,
    tok_labert,
    max_length: int = MAX_LEN,
    lemma_col: str = "lemma",
):
    rows = []
    for r in df.itertuples(index=False):
        # ---- Surface view ----
        tokens_surf, tidx_surf = _tokens_and_target_idx(r.left_context, r.target_word, r.right_context)
        span_x_surf = span_xlmr(tok_xlmr,  tokens_surf, tidx_surf, max_length=max_length)
        span_l_surf = span_labert(tok_labert, tokens_surf, tidx_surf, max_length=max_length)

        # ---- Lemma view (replace target by lemma; same left/right) ----
        target_lemma = re.sub(r"\d+$", "", getattr(r, lemma_col))
        tokens_lem, tidx_lem = _tokens_and_target_idx(r.left_context, target_lemma, r.right_context)
        span_x_lem = span_xlmr(tok_xlmr,  tokens_lem, tidx_lem, max_length=max_length)
        span_l_lem = span_labert(tok_labert, tokens_lem, tidx_lem, max_length=max_length)

        row_out = {
            **r._asdict(),

            # Surface view
            "tokens": tokens_surf,
            "target_idx": tidx_surf,
            "xmlr_wp_span": span_x_surf,
            "labert_wp_span": span_l_surf,
            "xmlr_wp_len": len(span_x_surf),
            "labert_wp_len": len(span_l_surf),

            # Lemma view
            "tokens_lemma": tokens_lem,
            "target_idx_lemma": tidx_lem,
            "xmlr_wp_span_lemma": span_x_lem,
            "labert_wp_span_lemma": span_l_lem,
            "xmlr_wp_len_lemma": len(span_x_lem),
            "labert_wp_len_lemma": len(span_l_lem),
        }
        rows.append(row_out)

    return pd.DataFrame(rows)

In [20]:
wsd_data_proc = add_nav_columns_after_merge(wsd_data, tokenizer_xlmr, tokenizer_labert)

In [21]:
import random

df = wsd_data_proc[wsd_data_proc["wsd_source"] == "semeval_wsd"]

# (1) only lemmas with ≥2 senses
n_senses = df.groupby("lemma")["sense_id"].nunique()
lemmas_with_2plus = n_senses[n_senses >= 2].index

# (2) each sense has at least 10 examples
per_sense_counts = df.groupby(["lemma", "sense_id"]).size()
min_per_lemma = per_sense_counts.groupby("lemma").min()
lemmas_with_min10 = min_per_lemma[min_per_lemma >= 5].index

# intersection
eligible_lemmas = lemmas_with_2plus.intersection(lemmas_with_min10)

lemma_holdout = set(random.sample(list(eligible_lemmas), 5))

print(f"Eligible lemmas: {len(eligible_lemmas)}")
print(f"Holdout lemmas ({len(lemma_holdout)}):", sorted(lemma_holdout))

Eligible lemmas: 16
Holdout lemmas (5): ['consilium', 'dux', 'fidelis', 'itero', 'sapientia']


In [22]:
lemma_holdout

{'consilium', 'dux', 'fidelis', 'itero', 'sapientia'}

In [23]:
semeval_holdout_proc = wsd_data_proc[(wsd_data_proc["wsd_source"] == "semeval_wsd") & (wsd_data_proc["lemma"].isin(lemma_holdout))]

In [24]:
semeval_holdout_proc.to_parquet("../data/wsd_data/semeval_holdout.parquet")

In [25]:
wsd_data_proc = wsd_data_proc[~wsd_data_proc["lemma"].isin(lemma_holdout)]

In [26]:
wsd_data_proc["sense_id"] = wsd_data_proc.apply(lambda row: f"{row['wsd_source']}_{row['sense_id']}", axis=1)

In [27]:
train_v1_proc = wsd_data_proc[wsd_data_proc["wsd_source"] == "labert_wsd"]
train_v2_proc = wsd_data_proc[wsd_data_proc["wsd_source"] == "semeval_wsd"]
train_v3_proc = wsd_data_proc[wsd_data_proc["wsd_source"].isin(["labert_wsd", "semeval_wsd"])]
train_v4_proc = wsd_data_proc[wsd_data_proc["wsd_source"].isin(["labert_wsd", "semeval_wsd", "silver_inter_wsd"])]

## Testing data shape etc.

In [28]:
import importlib, wsd_io
importlib.reload(wsd_io)

from wsd_io import *


from wsd_io import (
    WSDDataset,
    WiCPairDataset,
    WiCPairCollator,
    WiCPairModel,
    Collator,
    GroupedBatchSamplerWithReplacement,
    BalancedPKReplacementSampler,
    SupConLoss,
    pool_target_span,
)

# pull in everything the module defines (classes, functions, constants)
from wsd_io import *


In [29]:
wic_df = train_v4_proc  # e.g., labert_wsd + semeval_wsd only
wic_ds = WiCPairDataset(wic_df, pos_frac=0.5, pairs_per_epoch=128, seed=123)
wic_collate = wsd_io.WiCPairCollator(tokenizer_labert, model_kind="labert", use_lemma_view=True, max_len=160)

In [30]:
wic_dl = DataLoader(
    wic_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda batch: wic_collate(batch, rows=wic_ds.rows),
)

batch = next(iter(wic_dl))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, v.shape)

input_ids_1 torch.Size([32, 160])
attention_mask_1 torch.Size([32, 160])
input_ids_2 torch.Size([32, 160])
attention_mask_2 torch.Size([32, 160])
labels torch.Size([32])


In [31]:
# Take a small batch
batch = next(iter(wic_dl))

# How many to inspect
n_show = 3

for i in range(n_show):
    # Decode both sides
    text1 = tokenizer_labert.decode(batch["input_ids_1"][i], skip_special_tokens=True)
    text2 = tokenizer_labert.decode(batch["input_ids_2"][i], skip_special_tokens=True)
    label = batch["labels"][i].item()

    print("="*80)
    print(f"PAIR {i+1} | label={label}")
    print(f"USAGE 1: {text1}")
    print(f"USAGE 2: {text2}")

PAIR 1 | label=0
USAGE 1: facio_ non_ possum_ quin_ ad_ te_ mittam_
USAGE 2: demosthe nem_ si_ illa_ pronuntia re_ voluisset_ ornat e_ splendid e_ que_ facio_ potuisse_
PAIR 2 | label=1
USAGE 1: rare nter_ ueni o_ nimis_ pol_ inpu dente r_ serui s_ praest olar as_ ego_ puerum_ interea_ ancilla_ subd am_ lactan tem_ meae_ ne_ fame_ perb itat _ credit o_ cum_ illo c_ olli _
USAGE 2: expo liri _ ping i_ fing i_ et_ una_ bina e_ singulis_ quae_ datae_ nobis_ ancilla_ eae_ nos_ lava ndo_ elu endo_ operam_ dederunt_ agge run da que_ aqua_ sunt_ viri_
PAIR 3 | label=1
USAGE 1: ac_ volo_ inic ere_ trag ulam_ in_ nostrum_ senem_
USAGE 2: volo_ quidam_ iram_ in_ pectore_ moveri_ effer vesc ente_ circa_ cor_ sanguine_


In [32]:
def _safe_detok(tokenizer, ids, trim_len=None):
    """
    Detokenize a single sequence of ids for any tokenizer:
      - prefers .decode
      - else falls back to convert_ids_to_tokens + simple join/cleanup
    """
    ids = ids.tolist() if hasattr(ids, "tolist") else list(ids)
    if trim_len is not None:
        ids = ids[:int(trim_len)]

    # Fast path for HF tokenizers (e.g., XLM-R)
    dec = getattr(tokenizer, "decode", None)
    if callable(dec):
        return tokenizer.decode(ids, skip_special_tokens=True)

    # Fallback: token list → string
    toktok = getattr(tokenizer, "convert_ids_to_tokens", None)
    if callable(toktok):
        toks = toktok(ids)
        # strip known specials if tokenizer exposes ids
        specials = set()
        for name in ["pad_token_id", "cls_token_id", "bos_token_id", "eos_token_id", "sep_token_id"]:
            tid = getattr(tokenizer, name, None)
            if tid is not None:
                specials.add(tid)
        toks = [t for (t, tid) in zip(toks, ids) if tid not in specials and t is not None]

        # de-spm if needed (handle '▁' word boundary)
        s = " ".join(toks)
        s = s.replace("▁", " ")
        s = " ".join(s.split())
        return s

    # Last resort: just show ids
    return " ".join(map(str, ids))


def _trim_by_mask(ids, mask):
    L = int(mask.sum().item())
    return ids[:L]


def show_wic_examples_simple(dl, tokenizer, n=3):
    batch = next(iter(dl))
    B = int(batch["labels"].shape[0])
    n = min(n, B)

    for i in range(n):
        print("=" * 80)
        print(f"PAIR {i+1} | label={int(batch['labels'][i].item())}")

        ids1 = batch["input_ids_1"][i].cpu()
        m1   = batch["attention_mask_1"][i].cpu()
        ids2 = batch["input_ids_2"][i].cpu()
        m2   = batch["attention_mask_2"][i].cpu()

        ids1_trim = _trim_by_mask(ids1, m1)
        ids2_trim = _trim_by_mask(ids2, m2)

        s1 = _safe_detok(tokenizer, ids1_trim)
        s2 = _safe_detok(tokenizer, ids2_trim)
        print("DECODED 1 :", s1)
        print("DECODED 2 :", s2)

        # Also show token strings + spans if available
        toks1 = getattr(tokenizer, "convert_ids_to_tokens", lambda x: None)(ids1_trim.tolist()) or []
        toks2 = getattr(tokenizer, "convert_ids_to_tokens", lambda x: None)(ids2_trim.tolist()) or []
        span1 = list(batch["spans_1"][i])
        span2 = list(batch["spans_2"][i])

        if toks1:
            tokline1 = " ".join(toks1)
            print("SPAN_1   :", span1)
            print("TOKENS 1 :", tokline1)
        if toks2:
            tokline2 = " ".join(toks2)
            print("SPAN_2   :", span2)
            print("TOKENS 2 :", tokline2)


wic_df = train_v3_proc  # e.g., labert_wsd + semeval_wsd only
wic_ds = WiCPairDataset(wic_df, pos_frac=0.5, pairs_per_epoch=128, seed=123)
wic_collate = wsd_io.WiCPairCollator(tokenizer_labert, model_kind="labert", use_lemma_view=True, max_len=160)
wic_dl = DataLoader(
    wic_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda batch: wic_collate(batch, rows=wic_ds.rows),
)
show_wic_examples_simple(wic_dl, tokenizer_labert, n=3)

PAIR 1 | label=1
DECODED 1 : parvus_ sunt_ foris_ arma_ nisi_ est_ consilium_ domi_
DECODED 2 : in_ parvus_ quendam_ et_ angust um_ locum_ concludi _
SPAN_1   : [0]
TOKENS 1 : parvus_ sunt_ foris_ arma_ nisi_ est_ consilium_ domi_
SPAN_2   : [1]
TOKENS 2 : in_ parvus_ quendam_ et_ angust um_ locum_ concludi _
PAIR 2 | label=0
DECODED 1 : filio_ agri_ reliquit_ ei_ non_ magnum_ modus_
DECODED 2 : qui_ rebus_ infiniti s_ modus_ constitua nt_
SPAN_1   : [6]
TOKENS 1 : filio_ agri_ reliquit_ ei_ non_ magnum_ modus_
SPAN_2   : [4]
TOKENS 2 : qui_ rebus_ infiniti s_ modus_ constitua nt_
PAIR 3 | label=0
DECODED 1 : adversus_ fama_ rumore s_ que_ hominum_ si_ satis_ firm us_ steter is_
DECODED 2 : per_ omnem_ provinciam_ magnae_ atro ces_ que_ fama_ iban t_
SPAN_1   : [1]
TOKENS 1 : adversus_ fama_ rumore s_ que_ hominum_ si_ satis_ firm us_ steter is_
SPAN_2   : [7]
TOKENS 2 : per_ omnem_ provinciam_ magnae_ atro ces_ que_ fama_ iban t_


In [33]:
# same loader, just rebuild the collator with use_lemma_view=False
wic_collate_surface = WiCPairCollator(tokenizer_xlmr, "xlmr", use_lemma_view=False, max_len=160)
wic_dl_surface = DataLoader(wic_ds, batch_size=8, shuffle=True,
                            collate_fn=lambda b: wic_collate_surface(b, rows=wic_ds.rows))
show_wic_examples_simple(wic_dl_surface, tokenizer_xlmr, n=3)

PAIR 1 | label=1
DECODED 1 : valet ima summis mutare deus
DECODED 2 : illum salutavi post etiam jussi valere
SPAN_1   : [1]
TOKENS 1 : <s> ▁valet ▁ima ▁sum mis ▁muta re ▁de us </s>
SPAN_2   : [8, 9]
TOKENS 2 : <s> ▁illum ▁salut avi ▁post ▁etiam ▁jus si ▁vale re </s>
PAIR 2 | label=0
DECODED 1 : faciemus scrobes tribus pedibus altas
DECODED 2 : somno quibus est opus alto
SPAN_1   : [11, 12]
TOKENS 1 : <s> ▁faci e mus ▁sc robe s ▁tribu s ▁pedi bus ▁alta s </s>
SPAN_2   : [6]
TOKENS 2 : <s> ▁som no ▁quibus ▁est ▁opus ▁alto </s>
PAIR 3 | label=1
DECODED 1 : in educandis custodiendis que iis quae procreaverunt usque ad eum finem dum possint se ipsa defendere
DECODED 2 : ludus repertus et longorum operum finis
SPAN_1   : [16, 17]
TOKENS 1 : <s> ▁in ▁educa ndis ▁custodi endis ▁que ▁i is ▁quae ▁proc rea verunt ▁usque ▁ad ▁eum ▁fine m ▁dum ▁possi nt ▁se ▁ipsa ▁defender e </s>
SPAN_2   : [10]
TOKENS 2 : <s> ▁lud us ▁reper tus ▁et ▁long orum ▁oper um ▁finis </s>


## LaBERT - Train classfier head + projection only

In [34]:
import importlib, wsd_io
importlib.reload(wsd_io)

# pull in everything the module defines (classes, functions, constants)
from wsd_io import WiCPairModel, WiCPairIterableDataset, WiCPairCollator, freeze_all_encoder_params, unfreeze_encoder_layers, train_wic_model, unfreeze_last_n_layers, _find_transformer_blocks

In [36]:
wic_df = train_v4_proc  # labert + semeval (same-source enforced inside the dataset)

# Dataset / loader (as you have)
ds = WiCPairDataset(wic_df, pos_frac=0.5, pairs_per_epoch=40_000, seed=123)
collate = WiCPairCollator(tokenizer_labert, model_kind="labert", use_lemma_view=True, max_len=160)
dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0,
                collate_fn=lambda b: collate(b, rows=ds.rows))

# Model: pool from a mid-high layer (e.g., -3) and FREEZE encoder

D = model_labert.config.hidden_size  # typically 768

model_labert_head = WiCPairModel(
    encoder=model_labert,
    layer_idx=-4,
    proj_dim=D,             # same as encoder layer dim
    freeze_encoder=True
)

# Train head-only: give encoder either lr=0 or omit (it will be frozen anyway)
model_labert_head = train_wic_model(
    model_labert_head, dl,
    device="cuda",
    epochs=3,
    lr_head=1e-3,
    lr_encoder=0.0,          # explicit: encoder gets no updates
    entropy_reg=0.03,
    log_every=100,
    probe_first=True,
)

[optimizer] initial groups:
  group 0: lr=1.00e-03  wd=0.01  n_params=592,130
[ep 1] step 100/3750  loss=0.8774  ema=0.8774  mean(p)=0.757  lr=[0.00044]
[ep 1] step 200/3750  loss=0.8526  ema=0.8762  mean(p)=0.757  lr=[0.0008844444444444445]
[ep 1] step 300/3750  loss=0.7603  ema=0.8704  mean(p)=0.695  lr=[0.0009989130050423794]
[ep 1] step 400/3750  loss=0.7958  ema=0.8667  mean(p)=0.687  lr=[0.0009940000241095617]
[ep 1] step 500/3750  loss=0.8059  ema=0.8636  mean(p)=0.640  lr=[0.0009851658251629131]
[ep 1] step 600/3750  loss=0.8802  ema=0.8644  mean(p)=0.619  lr=[0.0009724805313165167]
[ep 1] step 700/3750  loss=0.6428  ema=0.8534  mean(p)=0.613  lr=[0.0009560448344738408]
[ep 1] step 800/3750  loss=0.6468  ema=0.8430  mean(p)=0.563  lr=[0.000935989196066837]
[ep 1] step 900/3750  loss=0.7076  ema=0.8363  mean(p)=0.573  lr=[0.0009124728114938094]
[ep 1] step 1000/3750  loss=0.6840  ema=0.8287  mean(p)=0.545  lr=[0.0008856823464760283]
[ep 1] step 1100/3750  loss=0.6680  ema=0.8206

In [37]:
import torch, os

save_dir = "../data/models/wic_labert_v1"
os.makedirs(save_dir, exist_ok=True)

torch.save({
    "proj": model_labert_head.proj.state_dict(),                        # projection weights
    "layer_idx": int(model_labert_head.layer_idx),                      # which layer to pool
    "logit_scale_raw": model_labert_head.logit_scale_raw.detach().cpu(),# calibration (pre-sigmoid)
    "scale_max": float(model_labert_head.scale_max),                    # bound for temperature
}, f"{save_dir}/wic_head_min.pt")

## LaBERT - Fine-tune encoder

In [38]:
@torch.no_grad()
def quick_eval(model, dl, device="cuda", batches=20):
    model.eval()
    n, correct, ps = 0, 0, []
    for i, batch in enumerate(dl):
        if i >= batches: break
        y = batch["labels"].to(device)
        for k in ("input_ids_1","attention_mask_1","input_ids_2","attention_mask_2"):
            batch[k] = batch[k].to(device, non_blocking=True)
        logits,_,_ = model(batch)
        p = torch.sigmoid(logits)
        pred = (p > 0.5).long()
        correct += (pred == y).sum().item()
        n += y.numel()
        ps.append(float(p.mean().item()))
    model.train()
    return {"acc": round(correct/max(1,n), 3), "mean_p": round(sum(ps)/max(1,len(ps)), 3)}

In [ ]:
# LaBERT - unfreeze last 3 layers

POOL_LAYER = -4          # you used -4 above; keep it consistent
N_LAST = 3               # try {1,2,3}; start with 2–3
LR_HEAD = 5e-4           # smaller than head-only stage to reduce drift
LR_ENC  = 2e-5           # gentle encoder LR (1e-5 .. 5e-5 are typical)

# rebuild dataset/collate/loader now
ds = WiCPairDataset(wic_df, pos_frac=0.5, pairs_per_epoch=40_000, seed=123)
collate = WiCPairCollator(tokenizer_labert, model_kind="labert", use_lemma_view=True, max_len=160)
dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0,
                collate_fn=lambda b: collate(b, rows=ds.rows))

fresh_encoder = AutoModel.from_pretrained(
    "/srv/models/latin-bert",
    output_hidden_states=True,
    output_attentions=False
)

# `model` = your trained head-only WiCPairModel from the previous run
D = fresh_encoder.config.hidden_size  # should equal your proj_dim from the frozen model

wic_ft = WiCPairModel(
    encoder=fresh_encoder,
    layer_idx=POOL_LAYER,
    proj_dim=D,                 # same dim as you trained: you used D earlier
    freeze_encoder=False,       # we'll decide what to unfreeze next
    init_logit_scale=float(model_labert_head.logit_scale.detach().cpu().item()),
    scale_max=float(model_labert_head.scale_max),
)

# copy head weights + calibration from the trained frozen model
wic_ft.proj.load_state_dict(model_labert_head.proj.state_dict(), strict=True)
with torch.no_grad():
    wic_ft.bias.copy_(model_labert_head.bias)
    wic_ft.logit_scale_raw.copy_(model_labert_head.logit_scale_raw)

# uses your module helpers
freeze_all_encoder_params(wic_ft.encoder)
unfreeze_last_n_layers(
    wic_ft.encoder, n_last=N_LAST,
    train_layernorm=True,   # often stabilizes
    train_pooler=False
)

wic_ft = train_wic_model(
    model=wic_ft,
    dataloader=dl,              # your existing loader
    device="cuda",
    epochs=3,

    # two-group LRs
    lr_head=LR_HEAD,
    lr_encoder=LR_ENC,

    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_frac=0.06,
    cosine_decay=True,
    amp=True,
    log_every=100,

    # mild regularization to avoid overconfident collapse
    entropy_reg=0.02,

    # quick health check every ~500 steps (optional)
    eval_fn=lambda m: quick_eval(m, dl, device="cuda", batches=30),
    eval_every=500,

    # sanity probe before training (will catch NaNs)
    probe_first=True,
)

In [65]:
save_dir = "../data/models/wic_labert_ft"
os.makedirs(save_dir, exist_ok=True)

# --- tiny payload for inference / embedding reuse ---
payload = {
    "proj_state_dict": wic_ft.proj.state_dict(),            # head weights
    "logit_scale_raw": float(wic_ft.logit_scale_raw.cpu()), # temperature
    "scale_max": float(wic_ft.scale_max),
    "bias": float(wic_ft.bias.detach().cpu()),
    "layer_idx": int(wic_ft.layer_idx),
    "proj_out_dim": (wic_ft.proj[0].out_features),
    "enc_hidden_size": int(wic_ft.encoder.config.hidden_size),
    "base_model_path": "/srv/models/latin-bert",
    "tokenizer_kind": "labert",
    "use_lemma_view": True,
    "max_len": 160,
}
torch.save(payload, f"{save_dir}/wic_head.pt")

# --- full checkpoint for exact reproduction (encoder + head) ---
torch.save(wic_ft.state_dict(), f"{save_dir}/wic_full_state_dict.pt")

# (optional) run metadata
with open(f"{save_dir}/train_args.json","w") as f:
    json.dump({
        "pool_layer": wic_ft.layer_idx,
        "lr_head": 5e-4,
        "lr_encoder": 2e-5,
        "epochs": 3,
        "entropy_reg": 0.02,
        "unfrozen_last_n": 3,  # or 6 if you do the wider run
    }, f, indent=2)

In [2]:
base_path = "/srv/models/latin-bert"
ft_ckpt   = "../data/models/wic_labert_ft/wic_full_state_dict.pt"
out_path  = "../data/models/latin-bert-wic-ft"

# 1) Start from base model
model = AutoModel.from_pretrained(
    base_path,
    output_hidden_states=True,
    output_attentions=False
)

# 2) Read WiCPairModel checkpoint safely
state = torch.load(ft_ckpt, map_location="cpu", weights_only=True)

# 3) Extract ONLY the transformer block weights as 'layer.*'
enc_block = {}
for k, v in state.items():
    if ".encoder.layer." in k:
        # examples seen: 'encoder.encoder.layer.9.*' or 'encoder.layer.9.*'
        kk = k.split(".encoder.layer.", 1)[1]   # -> '9.*'
        enc_block[f"layer.{kk}"] = v

# 4) Load into the encoder submodule
missing, unexpected = model.encoder.load_state_dict(enc_block, strict=False)
print("encoder missing:", missing)       # usually []
print("encoder unexpected:", unexpected) # usually []

# 5) Save as a proper HF dir
os.makedirs(out_path, exist_ok=True)
model.save_pretrained(out_path)
print("Saved fine-tuned encoder to:", out_path)

encoder missing: []
encoder unexpected: []
Saved fine-tuned encoder to: ../data/models/latin-bert-wic-ft


In [ ]:
# LaBERT - unfreeze last 6 layers
wic_df = train_v4_proc  # labert + semeval (same-source enforced inside the dataset)

# Dataset / loader (as you have)
ds = WiCPairDataset(wic_df, pos_frac=0.5, pairs_per_epoch=40_000, seed=123)
collate = WiCPairCollator(tokenizer_labert, model_kind="labert", use_lemma_view=True, max_len=160)
dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0,
                collate_fn=lambda b: collate(b, rows=ds.rows))


POOL_LAYER = -4          # you used -4 above; keep it consistent
N_LAST = 6               # try {1,2,3}; start with 2–3
LR_HEAD = 5e-4           # smaller than head-only stage to reduce drift
LR_ENC  = 2e-5           # gentle encoder LR (1e-5 .. 5e-5 are typical)

fresh_encoder = AutoModel.from_pretrained(
    "/srv/models/latin-bert",
    output_hidden_states=True,
    output_attentions=False
)

    # MUST match the encoder

# `model` = your trained head-only WiCPairModel from the previous run
D = fresh_encoder.config.hidden_size  # should equal your proj_dim from the frozen model

wic_ft = WiCPairModel(
    encoder=fresh_encoder,
    layer_idx=POOL_LAYER,
    proj_dim=D,                 # same dim as you trained: you used D earlier
    freeze_encoder=False,       # we'll decide what to unfreeze next
    init_logit_scale=float(model.logit_scale.detach().cpu().item()),
    scale_max=float(model.scale_max),
)

# copy head weights + calibration from the trained frozen model
wic_ft.proj.load_state_dict(model.proj.state_dict(), strict=True)
with torch.no_grad():
    wic_ft.bias.copy_(model.bias)
    wic_ft.logit_scale_raw.copy_(model.logit_scale_raw)

# uses your module helpers
freeze_all_encoder_params(wic_ft.encoder)
unfreeze_last_n_layers(
    wic_ft.encoder, n_last=N_LAST,
    train_layernorm=True,   # often stabilizes
    train_pooler=False
)

wic_ft = train_wic_model(
    model=wic_ft,
    dataloader=dl,              # your existing loader
    device="cuda",
    epochs=3,

    # two-group LRs
    lr_head=LR_HEAD,
    lr_encoder=LR_ENC,

    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_frac=0.06,
    cosine_decay=True,
    amp=True,
    log_every=100,

    # mild regularization to avoid overconfident collapse
    entropy_reg=0.02,

    # quick health check every ~500 steps (optional)
    eval_fn=lambda m: quick_eval(m, dl, device="cuda", batches=30),
    eval_every=500,

    # sanity probe before training (will catch NaNs)
    probe_first=True,
)


In [ ]:
# === SAFE SWITCH CHECKLIST ===
import gc, torch
torch.cuda.empty_cache(); gc.collect()
del fresh_encoder, model_labert_head, dl, ds, collate, wic_ft  # or any old model/loader vars you used
# (redefine tokenizer + collator + dataset + dataloader to match the new encoder family)

## XLMR - head only

In [35]:
wic_df = train_v4_proc  # labert + semeval (same-source enforced inside the dataset)

# Dataset / loader (as you have)
ds = WiCPairDataset(wic_df, pos_frac=0.5, pairs_per_epoch=40_000, seed=123)
collate = WiCPairCollator(tokenizer_xlmr, model_kind="xlmr", use_lemma_view=True, max_len=160)
dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0,
                collate_fn=lambda b: collate(b, rows=ds.rows))

# Model: pool from a mid-high layer (e.g., -3) and FREEZE encoder

D = model_labert.config.hidden_size  # typically 768

model_xlmr_head = WiCPairModel(
    encoder=model_xlmr,
    layer_idx=-4,
    proj_dim=D,             # same as encoder layer dim
    freeze_encoder=True
)

# Train head-only: give encoder either lr=0 or omit (it will be frozen anyway)
model_xlmr_head = train_wic_model(
    model_xlmr_head, dl,
    device="cuda",
    epochs=3,
    lr_head=1e-3,
    lr_encoder=0.0,          # explicit: encoder gets no updates
    entropy_reg=0.03,
    log_every=100,
    probe_first=True,
)

XLMRobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


[optimizer] initial groups:
  group 0: lr=1.00e-03  wd=0.01  n_params=592,130
[ep 1] step 100/3750  loss=1.0128  ema=1.0128  mean(p)=0.843  lr=[0.00044]
[ep 1] step 200/3750  loss=0.9138  ema=1.0078  mean(p)=0.793  lr=[0.0008844444444444445]
[ep 1] step 300/3750  loss=0.9509  ema=1.0050  mean(p)=0.750  lr=[0.0009989130050423794]
[ep 1] step 400/3750  loss=0.6661  ema=0.9881  mean(p)=0.699  lr=[0.0009940000241095617]
[ep 1] step 500/3750  loss=0.7366  ema=0.9755  mean(p)=0.696  lr=[0.0009851658251629131]
[ep 1] step 600/3750  loss=0.7850  ema=0.9660  mean(p)=0.622  lr=[0.0009724805313165167]
[ep 1] step 700/3750  loss=0.8642  ema=0.9609  mean(p)=0.663  lr=[0.0009560448344738408]
[ep 1] step 800/3750  loss=0.8182  ema=0.9537  mean(p)=0.667  lr=[0.000935989196066837]
[ep 1] step 900/3750  loss=0.7913  ema=0.9456  mean(p)=0.579  lr=[0.0009124728114938094]
[ep 1] step 1000/3750  loss=0.6626  ema=0.9315  mean(p)=0.575  lr=[0.0008856823464760283]
[ep 1] step 1100/3750  loss=0.8708  ema=0.9284

In [36]:
save_dir = "../data/models/wic_xlmr_v1"
os.makedirs(save_dir, exist_ok=True)

torch.save({
    "proj": model_xlmr_head.proj.state_dict(),                        # projection weights
    "layer_idx": int(model_xlmr_head.layer_idx),                      # which layer to pool
    "logit_scale_raw": model_xlmr_head.logit_scale_raw.detach().cpu(),# calibration (pre-sigmoid)
    "scale_max": float(model_xlmr_head.scale_max),                    # bound for temperature
}, f"{save_dir}/wic_head_min.pt")

## XLMR - encoder tuning

In [39]:
# XLMR - unfreeze last 3 layers

POOL_LAYER = -4          # you used -4 above; keep it consistent
N_LAST = 3               # try {1,2,3}; start with 2–3
LR_HEAD = 5e-4           # smaller than head-only stage to reduce drift
LR_ENC  = 2e-5           # gentle encoder LR (1e-5 .. 5e-5 are typical)

# rebuild dataset/collate/loader now
ds = WiCPairDataset(wic_df, pos_frac=0.5, pairs_per_epoch=40_000, seed=123)
collate = WiCPairCollator(tokenizer_xlmr, model_kind="xlmr", use_lemma_view=True, max_len=160)
dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0,
                collate_fn=lambda b: collate(b, rows=ds.rows))

fresh_encoder = AutoModel.from_pretrained(
    "xlm-roberta-base",
    output_hidden_states=True,
    output_attentions=False
)

# `model` = your trained head-only WiCPairModel from the previous run
D = fresh_encoder.config.hidden_size  # should equal your proj_dim from the frozen model

wic_ft = WiCPairModel(
    encoder=fresh_encoder,
    layer_idx=POOL_LAYER,
    proj_dim=D,                 # same dim as you trained: you used D earlier
    freeze_encoder=False,       # we'll decide what to unfreeze next
    init_logit_scale=float(model_xlmr_head.logit_scale.detach().cpu().item()),
    scale_max=float(model_xlmr_head.scale_max),
)

# copy head weights + calibration from the trained frozen model
wic_ft.proj.load_state_dict(model_xlmr_head.proj.state_dict(), strict=True)
with torch.no_grad():
    wic_ft.bias.copy_(model_xlmr_head.bias)
    wic_ft.logit_scale_raw.copy_(model_xlmr_head.logit_scale_raw)

# uses your module helpers
freeze_all_encoder_params(wic_ft.encoder)
unfreeze_last_n_layers(
    wic_ft.encoder, n_last=N_LAST,
    train_layernorm=True,   # often stabilizes
    train_pooler=False
)

wic_ft = train_wic_model(
    model=wic_ft,
    dataloader=dl,              # your existing loader
    device="cuda",
    epochs=3,

    # two-group LRs
    lr_head=LR_HEAD,
    lr_encoder=LR_ENC,

    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_frac=0.06,
    cosine_decay=True,
    amp=True,
    log_every=100,

    # mild regularization to avoid overconfident collapse
    entropy_reg=0.02,

    # quick health check every ~500 steps (optional)
    eval_fn=lambda m: quick_eval(m, dl, device="cuda", batches=30),
    eval_every=500,

    # sanity probe before training (will catch NaNs)
    probe_first=True,
)

[optimizer] initial groups:
  group 0: lr=5.00e-04  wd=0.01  n_params=592,130
  group 1: lr=2.00e-05  wd=0.01  n_params=21,292,800
[ep 1] step 100/3750  loss=0.7849  ema=0.7849  mean(p)=0.472  lr=[0.00022, 8.8e-06]
[ep 1] step 200/3750  loss=0.6653  ema=0.7789  mean(p)=0.501  lr=[0.00044222222222222227, 1.768888888888889e-05]
[ep 1] step 300/3750  loss=0.7294  ema=0.7764  mean(p)=0.510  lr=[0.0004994565025211897, 1.997826010084759e-05]
[ep 1] step 400/3750  loss=0.6816  ema=0.7717  mean(p)=0.521  lr=[0.0004970000120547808, 1.988000048219123e-05]
[ep 1] step 500/3750  loss=0.6333  ema=0.7648  mean(p)=0.528  lr=[0.0004925829125814566, 1.9703316503258265e-05]
[eval@500] {'acc': 0.565, 'mean_p': 0.588}
[ep 1] step 600/3750  loss=0.6848  ema=0.7608  mean(p)=0.517  lr=[0.00048624026565825834, 1.9449610626330336e-05]
[ep 1] step 700/3750  loss=0.6631  ema=0.7559  mean(p)=0.510  lr=[0.0004780224172369204, 1.9120896689476817e-05]
[ep 1] step 800/3750  loss=0.6659  ema=0.7514  mean(p)=0.501  lr=

In [40]:
save_dir = "../data/models/wic_xlmr_ft"
os.makedirs(save_dir, exist_ok=True)

# --- tiny payload for inference / embedding reuse ---
payload = {
    "proj_state_dict": wic_ft.proj.state_dict(),            # head weights
    "logit_scale_raw": float(wic_ft.logit_scale_raw.cpu()), # temperature
    "scale_max": float(wic_ft.scale_max),
    "bias": float(wic_ft.bias.detach().cpu()),
    "layer_idx": int(wic_ft.layer_idx),
    "proj_out_dim": (wic_ft.proj[0].out_features),
    "enc_hidden_size": int(wic_ft.encoder.config.hidden_size),
    "tokenizer_kind": "xlmr",
    "use_lemma_view": True,
    "max_len": 160,
}
torch.save(payload, f"{save_dir}/wic_head.pt")

# --- full checkpoint for exact reproduction (encoder + head) ---
torch.save(wic_ft.state_dict(), f"{save_dir}/wic_full_state_dict.pt")

# (optional) run metadata
with open(f"{save_dir}/train_args.json","w") as f:
    json.dump({
        "pool_layer": wic_ft.layer_idx,
        "lr_head": 5e-4,
        "lr_encoder": 2e-5,
        "epochs": 3,
        "entropy_reg": 0.02,
        "unfrozen_last_n": 3,  # or 6 if you do the wider run
    }, f, indent=2)